# ver14 Stage 1 - BiomedCLIP Internal Adapter Fine-tuning Fold 1

이 노트북은 `ver11`의 feature-level adapter probe와 `ver13`의 LoRA 5-Fold 결과 이후, 다음 질문을 확인하기 위한 실험입니다.

> BiomedCLIP visual encoder 내부 마지막 block에 작은 bottleneck adapter를 삽입하면, frozen feature 뒤에 adapter를 붙인 것보다 더 좋은 도메인 적응이 가능한가?

Stage 1 문제 정의는 이전과 동일합니다.

- 0: `NonDemented`
- 1: `Demented = VeryMildDemented + MildDemented + ModerateDemented`

기본 설정:

- Fold 1만 실행
- augmentation 없음
- BiomedCLIP 기존 weight 고정
- 마지막 visual block 2개에 adapter 삽입
- adapter와 classifier head만 학습

처음부터 augmentation을 넣지 않는 이유는 성능 최대화가 아니라 **원인 분리**입니다. 먼저 internal adapter 자체가 효과적인지 확인하고, 이후 별도 실험에서 augmentation을 추가해야 해석이 깔끔합니다.


## 1. 환경 점검 및 Seed 고정


In [1]:
import os
import sys
import time
import gc
import re
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version}")
print(f"torch version     : {torch.__version__}")
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"cuda available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name          : {torch.cuda.get_device_name(0)}")
else:
    print("GPU name          : CUDA GPU not available")
print(f"device            : {device}")


Python executable : C:\Users\user\anaconda3\envs\alzheimer\python.exe
Python version    : 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
torch version     : 2.12.0.dev20260408+cu128
torch.version.cuda: 12.8
cuda available    : True
GPU name          : NVIDIA GeForce RTX 5070
device            : cuda


## 2. 실험 설정


In [2]:
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")
OUTPUT_DIR = Path(r"C:\Users\user\alzheimer\patient_level_stage1_internal_adapter_fold1")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASSES = ["NonDemented"]
POSITIVE_CLASSES = ["VeryMildDemented", "MildDemented", "ModerateDemented"]
TASK_NAME = "non_vs_demented_biomedclip_internal_adapter_fold1"

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15
FOLDS_TO_RUN = [1]

# Internal adapter 설정
ADAPTER_LAST_VISUAL_BLOCKS = 2
ADAPTER_HIDDEN_DIM = 64
ADAPTER_DROPOUT = 0.1

# 첫 실험은 adapter 자체 효과를 보기 위해 augmentation 없이 둡니다.
TRAIN_AUG_MODE = "original"  # "original" 또는 "sepaug_4n"
ROTATION_DEGREES = 10
SHIFT_TRANSLATE = (0.05, 0.05)
ZOOM_SCALE = (0.9, 1.1)
DETERMINISTIC_AUGMENTATION = True

BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True

EPOCHS = 5
HEAD_LR = 5e-4
ADAPTER_LR = 5e-5
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
BALANCE_STRATEGY = "class_weight_sqrt"

# Screening 목적이므로 validation에서 민감도 우선 threshold를 선택합니다.
TARGET_VALIDATION_SENSITIVITY = 0.85
MIN_VALIDATION_SPECIFICITY = 0.65
THRESHOLD_GRID = np.round(np.arange(0.10, 0.7001, 0.025), 3)

RESULTS_PATH = OUTPUT_DIR / f"{TASK_NAME}_fold_results.csv"
SUMMARY_PATH = OUTPUT_DIR / f"{TASK_NAME}_summary.csv"

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"FOLDS_TO_RUN: {FOLDS_TO_RUN}")
print(f"TRAIN_AUG_MODE: {TRAIN_AUG_MODE}")
print(
    f"Internal adapter: last_blocks={ADAPTER_LAST_VISUAL_BLOCKS}, "
    f"hidden_dim={ADAPTER_HIDDEN_DIM}, dropout={ADAPTER_DROPOUT}"
)


DATA_ROOT : C:\Users\user\Desktop\alzheimer_dataset\Data
OUTPUT_DIR: C:\Users\user\alzheimer\patient_level_stage1_internal_adapter_fold1
MODEL_NAME: hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224
FOLDS_TO_RUN: [1]
TRAIN_AUG_MODE: original
Internal adapter: last_blocks=2, hidden_dim=64, dropout=0.1


## 3. 환자 단위 Manifest 생성

`ver9~ver13`과 동일하게 파일명에서 `OAS1_xxxx` 환자 ID를 추출합니다.
같은 환자의 slice는 반드시 같은 split에만 들어가야 하므로 Fold는 환자 단위로 나눕니다.


In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
selected_classes = NEGATIVE_CLASSES + POSITIVE_CLASSES

for class_name in selected_classes:
    target = 0 if class_name in NEGATIVE_CLASSES else 1
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), f"Class directory not found: {class_dir}"

    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, f"Patient ID를 추출할 수 없습니다: {image_path.name}"
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert not manifest.empty
assert manifest.groupby("patient_id")["target"].nunique().max() == 1
assert manifest.groupby("patient_id")["class_name"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        class_name=("class_name", "first"),
        image_count=("image_path", "count"),
    )
)

manifest_path = OUTPUT_DIR / f"{TASK_NAME}_all_images_manifest.csv"
patient_table_path = OUTPUT_DIR / f"{TASK_NAME}_patient_table.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
patient_table.to_csv(patient_table_path, index=False, encoding="utf-8-sig")

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
print("\n[Binary target patient counts]")
print(patient_table["target"].value_counts().sort_index())
print("\n[Original class patient counts]")
print(patient_table["class_name"].value_counts())


Images: 86,437
Patients: 347

[Binary target patient counts]
target
0    266
1     81
Name: count, dtype: int64

[Original class patient counts]
class_name
NonDemented         266
VeryMildDemented     58
MildDemented         21
ModerateDemented      2
Name: count, dtype: int64


## 4. BiomedCLIP 로드 및 feature dimension 확인


In [4]:
import open_clip


def load_biomedclip_model():
    print("BiomedCLIP 로드 중...")
    model, preprocess = open_clip.create_model_from_pretrained(MODEL_NAME)
    model = model.to(device)
    model.eval()

    model_device = next(model.parameters()).device
    print(f"model parameter device: {model_device}")
    assert model_device.type == device.type, (
        f"BiomedCLIP model is not on expected device. expected={device}, current={model_device}"
    )
    return model, preprocess


def infer_image_feature_dim(model, preprocess, sample_image_path):
    model.eval()
    image = Image.open(sample_image_path).convert("RGB")
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.inference_mode():
        with torch.amp.autocast(
            "cuda",
            enabled=(USE_AMP and torch.cuda.is_available()),
        ):
            features = model.encode_image(tensor)
    return int(features.shape[-1])


base_model_for_check, preprocess = load_biomedclip_model()
sample_image_path = manifest.iloc[0]["image_path"]
FEATURE_DIM = infer_image_feature_dim(base_model_for_check, preprocess, sample_image_path)
print(f"FEATURE_DIM: {FEATURE_DIM}")

del base_model_for_check
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


C:\Users\user\anaconda3\envs\alzheimer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BiomedCLIP 로드 중...


model parameter device: cuda:0
FEATURE_DIM: 512


## 5. Dataset 정의

기본값은 `TRAIN_AUG_MODE="original"`입니다.

이번 실험은 augmentation 효과가 아니라 internal adapter 효과를 확인하는 것이 목적입니다.
따라서 먼저 원본 이미지만 사용합니다.


In [5]:
class PatientImageDataset(Dataset):
    def __init__(
        self,
        dataframe,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=42,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess
        self.deterministic = deterministic
        self.seed = seed

        if aug_mode == "original":
            self.output_types = ["original"]
        elif aug_mode == "sepaug_4n":
            self.output_types = ["original", "rotation", "shift", "zoom"]
        else:
            raise ValueError(aug_mode)

        self.rotation = transforms.RandomRotation(ROTATION_DEGREES)
        self.shift = transforms.RandomAffine(degrees=0, translate=SHIFT_TRANSLATE)
        self.zoom = transforms.RandomAffine(degrees=0, scale=ZOOM_SCALE)

    def __len__(self):
        return len(self.df) * len(self.output_types)

    def _apply_augmentation(self, image, aug_type):
        if aug_type == "original":
            return image
        if aug_type == "rotation":
            return self.rotation(image)
        if aug_type == "shift":
            return self.shift(image)
        if aug_type == "zoom":
            return self.zoom(image)
        raise ValueError(aug_type)

    def __getitem__(self, index):
        multiplier = len(self.output_types)
        base_index = index // multiplier
        aug_index = index % multiplier
        aug_type = self.output_types[aug_index]
        row = self.df.iloc[base_index]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.deterministic:
            local_seed = self.seed + base_index * 1009 + aug_index * 9176
            state_py = random.getstate()
            state_np = np.random.get_state()
            state_torch = torch.random.get_rng_state()
            random.seed(local_seed)
            np.random.seed(local_seed % (2**32 - 1))
            torch.manual_seed(local_seed)
            image = self._apply_augmentation(image, aug_type)
            random.setstate(state_py)
            np.random.set_state(state_np)
            torch.random.set_rng_state(state_torch)
        else:
            image = self._apply_augmentation(image, aug_type)

        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )


## 6. Internal Adapter 모듈 정의

adapter는 기존 block의 출력을 작은 bottleneck network에 통과시켜 다시 더합니다.

```text
output = frozen_block(x)
output = output + adapter(output)
```

중요한 점:

- 기존 BiomedCLIP block weight는 고정합니다.
- adapter의 마지막 projection은 0으로 초기화합니다.
- 따라서 학습 시작 시점에는 원래 BiomedCLIP 출력과 거의 동일합니다.
- 학습되는 것은 adapter와 classifier head뿐입니다.


In [6]:
class BottleneckAdapter(nn.Module):
    def __init__(self, dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.up = nn.Linear(hidden_dim, dim)

        nn.init.kaiming_uniform_(self.down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class AdapterWrappedBlock(nn.Module):
    def __init__(self, block, dim, hidden_dim=64, dropout=0.1):
        super().__init__()
        self.block = block
        for parameter in self.block.parameters():
            parameter.requires_grad = False
        self.adapter = BottleneckAdapter(dim, hidden_dim, dropout)

    def forward(self, *args, **kwargs):
        output = self.block(*args, **kwargs)
        if isinstance(output, tuple):
            main = output[0]
            return (main + self.adapter(main), *output[1:])
        return output + self.adapter(output)


def infer_block_dim(block):
    for module in block.modules():
        if isinstance(module, nn.LayerNorm):
            shape = module.normalized_shape
            if isinstance(shape, int):
                return int(shape)
            if len(shape) > 0:
                return int(shape[-1])

    candidates = []
    for module in block.modules():
        if isinstance(module, nn.Linear):
            candidates.extend([module.in_features, module.out_features])
    if candidates:
        # ViT hidden dimension is usually the smaller repeated feature dimension.
        values, counts = np.unique(candidates, return_counts=True)
        return int(values[np.argmax(counts)])

    raise RuntimeError("Block hidden dimension을 추론할 수 없습니다.")


def get_visual_block_container(clip_model):
    visual = getattr(clip_model, "visual", None)
    candidates = []
    if visual is not None:
        candidates.extend([
            ("visual.trunk.blocks", lambda: visual.trunk.blocks),
            ("visual.transformer.resblocks", lambda: visual.transformer.resblocks),
            ("visual.blocks", lambda: visual.blocks),
        ])

    for name, getter in candidates:
        try:
            blocks = getter()
            if hasattr(blocks, "__len__") and len(blocks) > 0:
                return name, blocks
        except Exception:
            pass
    return None, None


def apply_internal_adapters(
    clip_model,
    n_last_blocks=2,
    hidden_dim=64,
    dropout=0.1,
):
    block_container_name, blocks = get_visual_block_container(clip_model)
    if blocks is None:
        raise RuntimeError("BiomedCLIP visual block container를 찾지 못했습니다.")

    n_last_blocks = min(n_last_blocks, len(blocks))
    wrapped = []

    for block_index in range(len(blocks) - n_last_blocks, len(blocks)):
        original_block = blocks[block_index]
        dim = infer_block_dim(original_block)
        wrapped_block = AdapterWrappedBlock(
            original_block,
            dim=dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )
        blocks[block_index] = wrapped_block
        wrapped.append({
            "name": f"{block_container_name}.{block_index}",
            "dim": dim,
            "hidden_dim": hidden_dim,
        })

    return wrapped


class BiomedCLIPInternalAdapterClassifier(nn.Module):
    def __init__(self, clip_model, feature_dim):
        super().__init__()
        self.clip_model = clip_model
        self.classifier = nn.Linear(feature_dim, 2)

    def forward(self, images):
        features = self.clip_model.encode_image(images)
        features = F.normalize(features.float(), dim=-1)
        return self.classifier(features)


def build_internal_adapter_classifier():
    clip_model, _ = load_biomedclip_model()

    for parameter in clip_model.parameters():
        parameter.requires_grad = False

    wrapped_blocks = apply_internal_adapters(
        clip_model,
        n_last_blocks=ADAPTER_LAST_VISUAL_BLOCKS,
        hidden_dim=ADAPTER_HIDDEN_DIM,
        dropout=ADAPTER_DROPOUT,
    )

    model = BiomedCLIPInternalAdapterClassifier(clip_model, FEATURE_DIM).to(device)
    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

    print(f"Internal adapter 적용 block 수: {len(wrapped_blocks)}")
    for item in wrapped_blocks:
        print(f"  - {item['name']} | dim={item['dim']} hidden={item['hidden_dim']}")

    return model, wrapped_blocks


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_parameters(model):
    return sum(p.numel() for p in model.parameters())


def trainable_state_dict(model):
    trainable_names = {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }
    return {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
        if name in trainable_names
    }


## 7. 평가 및 threshold 선택 함수


In [7]:
def safe_auroc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_auprc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(float(prob))
        grouped_labels[patient_id] = int(label)

    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds


def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": safe_auroc(y_true, y_prob),
        "auprc": safe_auprc(y_true, y_prob),
    }


def build_threshold_table(y_true, y_prob, threshold_grid):
    rows = []
    for threshold in threshold_grid:
        metrics = calculate_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append({
            "threshold": float(threshold),
            "distance_from_0_5": abs(float(threshold) - 0.5),
            **metrics,
        })
    return pd.DataFrame(rows)


def choose_sensitivity_first_threshold(
    y_true,
    y_prob,
    target_sensitivity=TARGET_VALIDATION_SENSITIVITY,
    min_specificity=MIN_VALIDATION_SPECIFICITY,
):
    table = build_threshold_table(y_true, y_prob, THRESHOLD_GRID)
    eligible = table[
        (table["sensitivity"] >= target_sensitivity)
        & (table["specificity"] >= min_specificity)
    ].copy()
    target_reached = not eligible.empty

    if not target_reached:
        eligible = table[table["specificity"] >= min_specificity].copy()
        if eligible.empty:
            eligible = table.copy()
        eligible = eligible[
            eligible["sensitivity"] == eligible["sensitivity"].max()
        ].copy()

    selected = eligible.sort_values(
        ["specificity", "precision", "distance_from_0_5"],
        ascending=[False, False, True],
    ).iloc[0]
    return float(selected["threshold"]), bool(target_reached), table


def evaluate_model(model, loader):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []

    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                probs = logits.softmax(dim=1)[:, 1]

            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probs.detach().cpu().numpy().tolist())

    return aggregate_patient_predictions(
        patient_ids,
        labels_all,
        probabilities_all,
        threshold=0.5,
    )[:3]


def make_class_weight(inner_train_patients):
    counts_series = inner_train_patients["target"].value_counts().sort_index()
    counts = np.array(
        [counts_series.get(0, 0), counts_series.get(1, 0)],
        dtype=np.float64,
    )
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None


## 8. Internal Adapter Fold 학습 함수


In [8]:
def make_loaders_for_fold(
    fold_number,
    outer_train_patients,
    outer_test_patients,
):
    run_seed = SEED + fold_number * 100
    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])

    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    train_dataset = PatientImageDataset(
        train_df,
        preprocess,
        aug_mode=TRAIN_AUG_MODE,
        deterministic=DETERMINISTIC_AUGMENTATION,
        seed=run_seed,
    )
    val_dataset = PatientImageDataset(
        val_df,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=run_seed,
    )
    test_dataset = PatientImageDataset(
        test_df,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=run_seed,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    return inner_train_patients, inner_val_patients, train_loader, val_loader, test_loader


def make_optimizer(model):
    head_params = []
    adapter_params = []
    other_params = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("classifier."):
            head_params.append(parameter)
        elif ".adapter." in name:
            adapter_params.append(parameter)
        else:
            other_params.append(parameter)

    groups = []
    if adapter_params:
        groups.append({"params": adapter_params, "lr": ADAPTER_LR})
    if other_params:
        groups.append({"params": other_params, "lr": ADAPTER_LR})
    if head_params:
        groups.append({"params": head_params, "lr": HEAD_LR})

    assert groups, "학습 가능한 parameter가 없습니다."
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def train_internal_adapter_one_fold(fold_number, outer_train_patients, outer_test_patients):
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients, train_loader, val_loader, test_loader = (
        make_loaders_for_fold(
            fold_number,
            outer_train_patients,
            outer_test_patients,
        )
    )

    model, wrapped_blocks = build_internal_adapter_classifier()
    optimizer = make_optimizer(model)
    class_weight = make_class_weight(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(USE_AMP and torch.cuda.is_available()),
    )

    trainable_params = count_trainable_parameters(model)
    total_params = count_total_parameters(model)
    trainable_ratio = trainable_params / max(total_params, 1)

    print(
        f"\n[Internal Adapter fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, "
        f"val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}"
    )
    print(
        f"trainable params: {trainable_params:,} / {total_params:,} "
        f"({trainable_ratio:.4%})"
    )

    best_state = None
    best_epoch = 0
    best_val_auroc = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, total = 0.0, 0
        epoch_start = time.time()

        for images, labels, _ in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                "cuda",
                enabled=(USE_AMP and torch.cuda.is_available()),
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        _, val_y_true, val_y_prob = evaluate_model(model, val_loader)
        val_metrics = calculate_metrics(val_y_true, val_y_prob, threshold=0.5)

        improved = val_metrics["auroc"] > best_val_auroc + EARLY_STOPPING_MIN_DELTA
        if improved:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = trainable_state_dict(model)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        elapsed = time.time() - epoch_start
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} "
            f"AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} | "
            f"{elapsed:.1f}s"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state, strict=False)
    model = model.to(device)

    val_ids, val_y_true, val_y_prob = evaluate_model(model, val_loader)
    selected_threshold, target_reached, threshold_table = (
        choose_sensitivity_first_threshold(val_y_true, val_y_prob)
    )

    test_ids, test_y_true, test_y_prob = evaluate_model(model, test_loader)
    default_test = calculate_metrics(test_y_true, test_y_prob, threshold=0.5)
    calibrated_test = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )

    checkpoint_path = CHECKPOINT_DIR / f"{TASK_NAME}_fold{fold_number}.pt"
    torch.save({
        "experiment": "internal_adapter",
        "fold": fold_number,
        "trainable_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "wrapped_blocks": wrapped_blocks,
        "config": {
            "model_name": MODEL_NAME,
            "feature_dim": FEATURE_DIM,
            "adapter_last_visual_blocks": ADAPTER_LAST_VISUAL_BLOCKS,
            "adapter_hidden_dim": ADAPTER_HIDDEN_DIM,
            "adapter_dropout": ADAPTER_DROPOUT,
            "train_aug_mode": TRAIN_AUG_MODE,
        },
    }, checkpoint_path)

    result = {
        "experiment": "internal_adapter",
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "selected_threshold": selected_threshold,
        "validation_target_reached": target_reached,
        "test_patients": len(test_ids),
        "default_sensitivity": default_test["sensitivity"],
        "default_specificity": default_test["specificity"],
        "default_f1": default_test["f1"],
        "calibrated_accuracy": calibrated_test["accuracy"],
        "calibrated_precision": calibrated_test["precision"],
        "calibrated_sensitivity": calibrated_test["sensitivity"],
        "calibrated_specificity": calibrated_test["specificity"],
        "calibrated_f1": calibrated_test["f1"],
        "calibrated_macro_f1": calibrated_test["macro_f1"],
        "auroc": calibrated_test["auroc"],
        "auprc": calibrated_test["auprc"],
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_ratio": trainable_ratio,
        "adapter_blocks": len(wrapped_blocks),
    }

    del model, optimizer, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


## 9. Internal Adapter 실행

기본값은 Fold 1만 실행합니다.

중간 결과 CSV가 있으면 이미 완료된 Fold는 건너뜁니다.


In [9]:
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

if RESULTS_PATH.exists():
    existing_results_df = pd.read_csv(RESULTS_PATH)
    all_results = existing_results_df.to_dict("records")
    completed_folds = {int(row["fold"]) for row in all_results}
    print(f"기존 결과 {len(all_results)}개를 로드했습니다.")
    print(f"완료된 folds: {sorted(completed_folds)}")
else:
    all_results = []
    completed_folds = set()
    print("기존 결과가 없습니다. 새로 시작합니다.")

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets),
    start=1,
):
    if fold_number not in FOLDS_TO_RUN:
        continue
    if fold_number in completed_folds:
        print(f"[SKIP] 이미 완료됨: fold {fold_number}")
        continue

    outer_train_patients = patient_table.iloc[
        outer_train_idx
    ].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[
        outer_test_idx
    ].reset_index(drop=True)

    print(
        f"\n===== INTERNAL ADAPTER OUTER FOLD {fold_number} =====\n"
        f"train patients={len(outer_train_patients)}, "
        f"test patients={len(outer_test_patients)}\n"
        f"test target counts="
        f"{outer_test_patients['target'].value_counts().sort_index().to_dict()}"
    )

    result = train_internal_adapter_one_fold(
        fold_number,
        outer_train_patients,
        outer_test_patients,
    )
    print("Test:", result)

    all_results.append(result)
    completed_folds.add(fold_number)

    temp_path = RESULTS_PATH.with_suffix(".tmp.csv")
    pd.DataFrame(all_results).to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig",
    )
    temp_path.replace(RESULTS_PATH)
    print(f"중간 결과 저장: {RESULTS_PATH}")

results_df = pd.DataFrame(all_results)
display(results_df.sort_values("fold"))


기존 결과가 없습니다. 새로 시작합니다.

===== INTERNAL ADAPTER OUTER FOLD 1 =====
train patients=277, test patients=70
test target counts={0: 54, 1: 16}
BiomedCLIP 로드 중...
model parameter device: cuda:0
Internal adapter 적용 block 수: 2
  - visual.trunk.blocks.10 | dim=768 hidden=64
  - visual.trunk.blocks.11 | dim=768 hidden=64

[Internal Adapter fold 1] inner train=235, val=42, outer test=70
trainable params: 199,298 / 196,102,019 (0.1016%)
Epoch 01/5 | loss 0.3704 | val AUROC 0.8781 AUPRC 0.6294 | early 0/2 | 342.1s
Epoch 02/5 | loss 0.2786 | val AUROC 0.8969 AUPRC 0.6554 | early 0/2 | 315.6s
Epoch 03/5 | loss 0.2281 | val AUROC 0.9000 AUPRC 0.7201 | early 0/2 | 316.2s
Epoch 04/5 | loss 0.1832 | val AUROC 0.9125 AUPRC 0.7314 | early 0/2 | 318.6s
Epoch 05/5 | loss 0.1452 | val AUROC 0.9156 AUPRC 0.7964 | early 0/2 | 312.2s
Test: {'experiment': 'internal_adapter', 'fold': 1, 'best_epoch': 5, 'val_auroc': 0.915625, 'selected_threshold': 0.325, 'validation_target_reached': True, 'test_patients': 70, 'defa

,experiment,fold,best_epoch,val_auroc,selected_threshold,validation_target_reached,test_patients,default_sensitivity,default_specificity,default_f1,...,calibrated_sensitivity,calibrated_specificity,calibrated_f1,calibrated_macro_f1,auroc,auprc,trainable_params,total_params,trainable_ratio,adapter_blocks
0,internal_adapter,1,5,0.915625,0.325,True,70,0.75,0.87037,0.685714,...,0.9375,0.777778,0.697674,0.781827,0.90162,0.641874,199298,196102019,0.001016,2


## 10. Internal Adapter 결과 요약 및 기존 실험 비교

Fold 1 기준 주요 비교 대상:

`ver11` adapter probe Fold 1:

- sensitivity: 0.875
- specificity: 0.833
- F1: 0.718
- AUROC: 0.907
- AUPRC: 0.710

`ver13` LoRA stable Fold 1:

- sensitivity: 0.938
- specificity: 0.796
- F1: 0.714
- AUROC: 0.895
- AUPRC: 0.622

internal adapter가 이 둘보다 좋아지면 encoder 내부에 adapter를 넣는 PEFT가 의미 있다는 뜻입니다.
비슷하거나 낮으면 현재 데이터에서는 feature-level adapter probe가 더 안정적인 선택입니다.


In [10]:
assert RESULTS_PATH.exists(), f"결과 CSV가 아직 없습니다: {RESULTS_PATH}"
results_df = pd.read_csv(RESULTS_PATH)

metric_columns = [
    "default_sensitivity",
    "default_specificity",
    "default_f1",
    "calibrated_accuracy",
    "calibrated_precision",
    "calibrated_sensitivity",
    "calibrated_specificity",
    "calibrated_f1",
    "calibrated_macro_f1",
    "auroc",
    "auprc",
    "selected_threshold",
    "trainable_params",
    "trainable_ratio",
    "adapter_blocks",
]

summary_rows = []
for experiment_name, group in results_df.groupby("experiment"):
    row = {
        "experiment": experiment_name,
        "folds": group["fold"].nunique(),
        "target_reached_folds": int(group["validation_target_reached"].sum()),
    }
    for metric in metric_columns:
        row[f"{metric}_mean"] = group[metric].mean()
        if group["fold"].nunique() > 1:
            row[f"{metric}_std"] = group[metric].std(ddof=1)
        else:
            row[f"{metric}_std"] = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("experiment")
summary_df.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

display(results_df.sort_values("fold"))
display(summary_df)
print(f"Summary saved: {SUMMARY_PATH}")

foundation_summary_path = Path(r"C:\Users\user\alzheimer\patient_level_stage1_foundation\non_vs_demented_biomedclip_summary.csv")
if foundation_summary_path.exists():
    print("\n[ver11 foundation model summary]")
    display(pd.read_csv(foundation_summary_path))
else:
    print(f"ver11 summary not found: {foundation_summary_path}")

lora_summary_path = Path(r"C:\Users\user\alzheimer\patient_level_stage1_lora_5fold_stable\non_vs_demented_biomedclip_lora_5fold_stable_summary.csv")
if lora_summary_path.exists():
    print("\n[ver13 LoRA stable summary]")
    display(pd.read_csv(lora_summary_path))
else:
    print(f"ver13 LoRA summary not found: {lora_summary_path}")

cnn_summary_path = Path(r"C:\Users\user\alzheimer\patient_level_stage1\sensitivity_first\sensitivity_target_summary.csv")
if cnn_summary_path.exists():
    print("\n[ver10 CNN sensitivity-first baseline]")
    display(pd.read_csv(cnn_summary_path))
else:
    print(f"ver10 CNN summary not found: {cnn_summary_path}")


,experiment,fold,best_epoch,val_auroc,selected_threshold,validation_target_reached,test_patients,default_sensitivity,default_specificity,default_f1,...,calibrated_sensitivity,calibrated_specificity,calibrated_f1,calibrated_macro_f1,auroc,auprc,trainable_params,total_params,trainable_ratio,adapter_blocks
0,internal_adapter,1,5,0.915625,0.325,True,70,0.75,0.87037,0.685714,...,0.9375,0.777778,0.697674,0.781827,0.90162,0.641874,199298,196102019,0.001016,2


,experiment,folds,target_reached_folds,default_sensitivity_mean,default_sensitivity_std,default_specificity_mean,default_specificity_std,default_f1_mean,default_f1_std,calibrated_accuracy_mean,...,auprc_mean,auprc_std,selected_threshold_mean,selected_threshold_std,trainable_params_mean,trainable_params_std,trainable_ratio_mean,trainable_ratio_std,adapter_blocks_mean,adapter_blocks_std
0,internal_adapter,1,1,0.75,NaN,0.87037,NaN,0.685714,NaN,0.814286,...,0.641874,NaN,0.325,NaN,199298.0,NaN,0.001016,NaN,2.0,NaN


Summary saved: C:\Users\user\alzheimer\patient_level_stage1_internal_adapter_fold1\non_vs_demented_biomedclip_internal_adapter_fold1_summary.csv

[ver11 foundation model summary]


,experiment,folds,target_reached_folds,default_sensitivity_mean,default_sensitivity_std,default_specificity_mean,default_specificity_std,default_f1_mean,default_f1_std,calibrated_accuracy_mean,...,calibrated_macro_f1_mean,calibrated_macro_f1_std,auroc_mean,auroc_std,auprc_mean,auprc_std,selected_threshold_mean,selected_threshold_std,trainable_params_mean,trainable_params_std
0,adapter_probe,5,5,0.700735,0.174692,0.860867,0.028796,0.644126,0.122376,0.838551,...,0.802147,0.038409,0.901043,0.039372,0.694710,0.086107,0.40,0.055902,133762.0,0.0
1,linear_probe,5,5,0.257353,0.251227,0.962264,0.040025,0.311026,0.291491,0.801035,...,0.760550,0.041824,0.897991,0.033547,0.687036,0.094032,0.36,0.037914,1026.0,0.0
2,zero_shot,5,0,0.222794,0.095997,0.778057,0.031984,0.225454,0.082200,0.648447,...,0.498839,0.048995,0.485927,0.072367,0.268011,0.066443,0.50,0.000000,0.0,0.0



[ver13 LoRA stable summary]


,experiment,folds,target_reached_folds,default_sensitivity_mean,default_sensitivity_std,default_specificity_mean,default_specificity_std,default_f1_mean,default_f1_std,calibrated_accuracy_mean,...,auprc_mean,auprc_std,selected_threshold_mean,selected_threshold_std,trainable_params_mean,trainable_params_std,trainable_ratio_mean,trainable_ratio_std,lora_layers_mean,lora_layers_std
0,lora,5,5,0.652941,0.18221,0.879665,0.043538,0.629249,0.113072,0.824141,...,0.672804,0.067742,0.405,0.103682,74754.0,0.0,0.000381,0.0,4.0,0.0



[ver10 CNN sensitivity-first baseline]


,target_sensitivity,target_reached_folds,median_threshold,mean_threshold,mean_test_sensitivity,std_test_sensitivity,mean_test_specificity,std_test_specificity,mean_test_precision,mean_test_f1,mean_test_macro_f1,mean_test_auroc,mean_test_auprc
0,0.80,5,0.475,0.455,0.738235,0.121752,0.887212,0.035311,0.667983,0.698486,0.800466,0.912233,0.733636
1,0.85,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636
2,0.90,4,0.450,0.410,0.775735,0.114857,0.857233,0.045003,0.627273,0.690341,0.790372,0.912233,0.733636


## 11. 결과 해석 가이드

internal adapter 결과는 다음 기준으로 해석합니다.

1. **adapter probe보다 좋다**
   - feature 뒤쪽 adapter만으로는 부족했고, visual encoder 내부 표현을 task에 맞게 살짝 조정하는 것이 효과적이었다는 뜻입니다.

2. **LoRA보다 좋다**
   - low-rank update보다 bottleneck adapter가 이 MRI task에는 더 적합했을 수 있습니다.

3. **둘보다 낮다**
   - 작은 데이터에서는 encoder 내부를 건드리는 방식보다 frozen feature 위의 adapter probe가 더 안정적이라는 결론이 됩니다.

Fold 1이 좋아도 바로 결론을 내리지 말고, 그 다음에는 같은 설정으로 5-Fold 확장을 해야 합니다.
